In [116]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [117]:
df = pd.read_csv("../data/raw/taxi_trip_pricing.csv")


# --------------------
# 1. Basic Cleaning
# --------------------
# Drop rows where target is mis


In [118]:
df = df.dropna(subset=["Trip_Price"])

In [119]:
# Convert numeric columns

In [120]:
num_cols = [
    "Trip_Distance_km", "Passenger_Count",
    "Base_Fare", "Per_Km_Rate", "Per_Minute_Rate",
    "Trip_Duration_Minutes"
]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")



# --------------------
# 2. Handle Missing Values
# --------------------
# Numeric: fill with median


In [121]:

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical: fill with "Unknown"

In [122]:
cat_cols = [
    "Time_of_Day", "Day_of_Week",
    "Traffic_Conditions", "Weather"
]


# --------------------
# 3. Feature Engineering
# --------------------


In [123]:

#Price components
df["Distance_Cost"] = df["Trip_Distance_km"] * df["Per_Km_Rate"]
df["Time_Cost"] = df["Trip_Duration_Minutes"] * df["Per_Minute_Rate"]

# Speed feature
df["Avg_Speed"] = df["Trip_Distance_km"] / (df["Trip_Duration_Minutes"] / 60 + 1e-5)

# Peak indicator
df["Is_Peak"] = df["Time_of_Day"].isin(["Morning", "Evening"]).astype(int)

# Weekend flag
df["Is_Weekend"] = (df["Day_of_Week"] == "Weekend").astype(int)

# High traffic flag
df["High_Traffic"] = (df["Traffic_Conditions"] == "High").astype(int)

# Bad weather flag
df["Bad_Weather"] = df["Weather"].isin(["Rain", "Snow"]).astype(int)



# --------------------
# 4. Prepare Features
# ------------------


In [124]:

target = "Trip_Price"

features = df.drop(columns=[target])
y = df[target]


# Updated column groups
num_features = features.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = features.select_dtypes(include=["object"]).columns.tolist()


C:\Users\3r7mef\AppData\Local\Temp\ipykernel_26384\127616141.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = features.select_dtypes(include=["object"]).columns.tolist()



# --------------------
# 5. Preprocessing Pipeline
# --------------------


In [125]:

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

# Final pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

# Transform features
X = pipeline.fit_transform(features)



# --------------------
# 6. Output
# --------------------

In [126]:

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (951, 29)
Target shape: (951,)


# --------------------
# Save pipeline
# --------------------

In [127]:
from pathlib import Path
import joblib

# Save processed dataframe
output_path = Path("../data/prepped/taxi_trip_pricing_prepped.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

# Save pipeline
pipeline_path = Path("../models/preprocessing_pipeline.joblib")
pipeline_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, pipeline_path)

['..\\models\\preprocessing_pipeline.joblib']

# Train

In [128]:
import optuna
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# --------------------
# Split data
# --------------------
X_train, X_test, y_train, y_test = train_test_split(
    features,
    y,
    test_size=0.2,
    random_state=42
)

# --------------------
# Define objective function
# --------------------
def objective(trial):

    model = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 200, 800),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.01, 10.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.01, 10.0, log=True),
        random_state=42
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    preds = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return rmse

# --------------------
# Run optimisation
# --------------------
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

# --------------------
# Best results
# --------------------
print("✅ Best RMSE:", study.best_value)
print("✅ Best params:", study.best_params)

[I 2026-05-08 13:09:51,252] A new study created in memory with name: no-name-1c9efcea-ed23-479c-bf40-9c107ea4c1b8
[I 2026-05-08 13:09:51,831] Trial 0 finished with value: 16.608633605522765 and parameters: {'n_estimators': 213, 'max_depth': 4, 'learning_rate': 0.011017365888843552, 'subsample': 0.8650571292406357, 'colsample_bytree': 0.9076323168329102, 'reg_lambda': 2.270850725386375, 'reg_alpha': 0.01239891207447036}. Best is trial 0 with value: 16.608633605522765.
[I 2026-05-08 13:09:53,573] Trial 1 finished with value: 12.1435843604949 and parameters: {'n_estimators': 536, 'max_depth': 7, 'learning_rate': 0.031971699612924495, 'subsample': 0.8513611005700672, 'colsample_bytree': 0.7877781068462932, 'reg_lambda': 0.018084346494642647, 'reg_alpha': 0.20600251102703593}. Best is trial 1 with value: 12.1435843604949.
[I 2026-05-08 13:09:54,948] Trial 2 finished with value: 11.498955572817016 and parameters: {'n_estimators': 391, 'max_depth': 8, 'learning_rate': 0.1010248525140787, 'sub

✅ Best RMSE: 10.37468270767609
✅ Best params: {'n_estimators': 629, 'max_depth': 8, 'learning_rate': 0.11230831157469905, 'subsample': 0.7872284773184578, 'colsample_bytree': 0.8249329973487373, 'reg_lambda': 0.016160892746754755, 'reg_alpha': 0.010243118371406412}


In [129]:
#save params
import json
from pathlib import Path

params_path = Path("../models/best_params.json")

params_path.parent.mkdir(exist_ok=True)

with open(params_path, "w") as f:
    json.dump(study.best_params, f, indent=4)

print("✅ Best params saved!")

✅ Best params saved!
